# ai03b Task Solutions: Build, Evaluate, and Visualize Decision Tree Models

**INSTRUCTOR SOLUTIONS - DO NOT DISTRIBUTE**

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt

print("✓ Libraries imported")

In [ ]:
titanicData = pd.read_csv('Titanic Dataset.csv')
columnsNeeded = ['pclass', 'survived', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']
titanicData = titanicData[columnsNeeded]
titanicData = titanicData.dropna()
titanicData = pd.get_dummies(titanicData, columns=['sex', 'embarked'], drop_first=True)

print(f"✓ Data loaded and cleaned: {titanicData.shape}")

In [ ]:
X = titanicData.drop('survived', axis=1)
y = titanicData['survived']

print(f"Features (X) shape: {X.shape}")
print(f"Target (y) shape: {y.shape}")
print(f"Survival distribution: {y.value_counts().to_dict()}")

In [ ]:
xTrain, xTest, yTrain, yTest = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set: {xTrain.shape[0]} rows")
print(f"Testing set: {xTest.shape[0]} rows")
print(f"Train/Test split: {len(xTrain)/(len(xTrain)+len(xTest)):.1%} / {len(xTest)/(len(xTrain)+len(xTest)):.1%}")

In [ ]:
model = DecisionTreeClassifier(max_depth=4, random_state=42)
model.fit(xTrain, yTrain)

print("✓ Model trained successfully!")
print(f"Model max_depth: {model.max_depth}")
print(f"Number of features used: {model.n_features_in_}")

In [ ]:
predictions = model.predict(xTest)

print(f"✓ Predictions made for {len(predictions)} passengers")
print(f"Predicted survivors: {sum(predictions)}")
print(f"Predicted non-survivors: {len(predictions) - sum(predictions)}")
print(f"\nFirst 10 predictions: {predictions[:10]}")
print(f"First 10 actual values: {yTest.values[:10]}")

In [ ]:
accuracy = accuracy_score(yTest, predictions)

print("="*60)
print(f"ACCURACY: {accuracy:.4f} ({accuracy*100:.2f}%)")
print("="*60)
print(f"Correct predictions: {int(accuracy*len(yTest))}/{len(yTest)}")
print(f"Incorrect predictions: {len(yTest) - int(accuracy*len(yTest))}/{len(yTest)}")

In [ ]:
print(classification_report(yTest, predictions, target_names=['Did Not Survive', 'Survived']))

In [ ]:
plt.figure(figsize=(20, 10))
plot_tree(model, feature_names=X.columns, class_names=['Did Not Survive', 'Survived'], 
          filled=True, rounded=True, fontsize=10)
plt.title("Decision Tree for Titanic Survival Prediction")
plt.tight_layout()
plt.show()

print("✓ Tree visualization complete")

In [ ]:
importances = model.feature_importances_

featureImportance = list(zip(X.columns, importances))
featureImportance.sort(key=lambda x: x[1], reverse=True)

print("Feature Importance (highest to lowest):")
print("="*50)
for feature, importance in featureImportance:
    bar = '█' * int(importance * 40)
    print(f"{feature:15} {importance:6.4f} {bar}")

In [ ]:
plt.figure(figsize=(10, 6))
features = [f[0] for f in featureImportance]
importances_sorted = [f[1] for f in featureImportance]
plt.bar(features, importances_sorted)
plt.xlabel('Feature')
plt.ylabel('Importance')
plt.title('Feature Importance for Titanic Survival')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
newPassenger = pd.DataFrame({
    'pclass': [1],
    'age': [30],
    'sibsp': [0],
    'parch': [0],
    'fare': [200],
    'sex_male': [0],
    'embarked_Q': [0],
    'embarked_S': [1]
})

prediction = model.predict(newPassenger)[0]
probability = model.predict_proba(newPassenger)[0]

print(f"Prediction: {'Survived' if prediction == 1 else 'Did not survive'}")
print(f"Confidence: {max(probability)*100:.2f}%")

In [ ]:
max_depths = [2, 3, 4, 5, 6, 7]
results = []

for depth in max_depths:
    tempModel = DecisionTreeClassifier(max_depth=depth, random_state=42)
    tempModel.fit(xTrain, yTrain)
    
    trainPred = tempModel.predict(xTrain)
    testPred = tempModel.predict(xTest)
    
    trainAcc = accuracy_score(yTrain, trainPred)
    testAcc = accuracy_score(yTest, testPred)
    
    results.append({
        'max_depth': depth,
        'train_accuracy': trainAcc,
        'test_accuracy': testAcc
    })

print("max_depth | Train Accuracy | Test Accuracy | Difference")
print("-" * 60)
for result in results:
    diff = result['train_accuracy'] - result['test_accuracy']
    print(f"    {result['max_depth']}    |    {result['train_accuracy']:.4f}     |   {result['test_accuracy']:.4f}    |  {diff:+.4f}")